# Sequence-to-Sequence (Summarization & Translation)

> Encoder-decoder models for text generation tasks

This notebook shows how to fine-tune T5, BART, and other seq2seq models with blurr.

In [ ]:
#| default_exp text.seq2seq

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from blurr.text.seq2seq import Seq2SeqLearner

## Summarization with T5

### From a DataFrame

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'article': [
        'The quick brown fox jumps over the lazy dog. The fox was very fast.',
        'Scientists discovered a new deep-sea fish species in the Pacific Ocean.',
        'The stock market rose sharply after the Federal Reserve announced new policies.',
        'A new study shows exercise can significantly improve mental health outcomes.',
        'The city council voted to approve construction of new parks and rec centers.',
        'Researchers found that coffee consumption may lower the risk of diseases.',
    ],
    'summary': [
        'A fox jumps over a dog.',
        'New deep-sea fish species found.',
        'Stock market rises after Fed announcement.',
        'Exercise improves mental health.',
        'City council approves new parks.',
        'Coffee may reduce disease risk.',
    ],
})

learn = Seq2SeqLearner.from_df(
    df,
    'google/t5-efficient-tiny',
    input_col='article',
    target_col='summary',
    prefix='summarize: ',  # T5 needs task prefix
    max_source_length=128,
    max_target_length=64,
    bs=2,
)

In [ ]:
learn.fit_one_cycle(1, 2e-4)

### Text Generation

In [ ]:
results = learn.blurr_generate(
    'The president announced new trade agreements with several Asian countries.',
    prefix='summarize: ',
    max_new_tokens=64,
    num_beams=4,
)
print(results[0]['generated_text'])

## Generation Parameters

blurr exposes all HuggingFace generation parameters:

```python
learn.blurr_generate(
    text,
    max_new_tokens=128,   # Maximum tokens to generate
    num_beams=4,          # Beam search width (1 = greedy)
    temperature=1.0,      # Sampling temperature (via **gen_kwargs)
    top_p=0.9,            # Nucleus sampling (via **gen_kwargs)
    no_repeat_ngram_size=3,  # Prevent repetition
)
```

## ROUGE Metrics

ROUGE metrics (rouge1, rouge2, rougeL) are automatically computed during validation when the `rouge-score` package is installed.

## Supported Models

Any HuggingFace encoder-decoder model works:
- **T5** / **mT5** / **Flan-T5** (text-to-text)
- **BART** / **mBART** (denoising autoencoder)
- **Pegasus** (abstractive summarization)
- **MarianMT** (translation)
- And more...